# MiniMax-H3-Image Dual-GPU (2x NVIDIA Tesla T4) — Full `/tmp` Setup

Notebook ini menjalankan **ComfyUI + MiniMax-H3-Image (W4A8 Diffusion + Qwen3-VL 32B NVFP4 AWQ + FP16 VAE)** sepenuhnya di **`/tmp`** (tanpa memakai `/kaggle/working`).

### Arsitektur Split Dual-GPU (`run_dual_gpu.bat` + `ComfyUI-MultiGPU`)
- **Flags ComfyUI**: `--listen 0.0.0.0 --port 8188 --cuda-device all --preview-method auto --enable-cors-header "*"` (`--enable-cors-header "*"` mencegah HTTP 403 `Sec-Fetch-Site: cross-site` saat link dibuka dari Kaggle/Cloudflare)
- **`cuda:0` (GPU0 - 15 GB)**: Text Encoder (`qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors`)
- **`cuda:1` (GPU1 - 15 GB)**: Diffusion UNET (`minimax_h3_image_pruned_w4a8.safetensors`) + VAE (`minimax_h3_image_vae_fp16.safetensors`)
- **Workflows otomatis terpasang di `/tmp/ComfyUI/user/default/workflows/`**:
  - `minimax_h3_image_t2i_dual_gpu.json`
  - `minimax_h3_image_edit_dual_gpu.json`

In [ ]:
import os
import shutil
import subprocess
import urllib.request

os.chdir("/tmp")
os.environ["TMPDIR"] = "/tmp"
os.environ["PIP_CACHE_DIR"] = "/tmp/pip_cache"

COMFY_ROOT = "/tmp/ComfyUI"
CUSTOM_NODES = os.path.join(COMFY_ROOT, "custom_nodes")
MANAGER_DIR = os.path.join(CUSTOM_NODES, "ComfyUI-Manager")
MULTIGPU_DIR = os.path.join(CUSTOM_NODES, "ComfyUI-MultiGPU")


def run(cmd, cwd="/tmp"):
    print(f"+ {cmd}", flush=True)
    subprocess.run(cmd, shell=True, check=True, cwd=cwd)


print("=== Cek GPU & Resource Container ===")
NVIDIA_SMI = shutil.which("nvidia-smi") or "/opt/bin/nvidia-smi"
run(f"{NVIDIA_SMI} --query-gpu=index,name,memory.total,memory.free --format=csv,noheader")
usage = shutil.disk_usage("/tmp")
print(f"/tmp free: {usage.free / (1024**3):.1f} GiB")

if shutil.which("aria2c") is None:
    run("apt-get update -qq && apt-get install -y -qq aria2")

if not os.path.exists(os.path.join(COMFY_ROOT, "main.py")):
    run(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_ROOT}")
else:
    print("ComfyUI sudah ada di /tmp/ComfyUI, skip clone.")

os.makedirs(CUSTOM_NODES, exist_ok=True)
if not os.path.exists(MANAGER_DIR):
    run(f"git clone --depth 1 https://github.com/Comfy-Org/ComfyUI-Manager.git {MANAGER_DIR}")
else:
    print("ComfyUI-Manager sudah ada, skip clone.")

if not os.path.exists(MULTIGPU_DIR):
    run(f"git clone --depth 1 https://github.com/pollockjj/ComfyUI-MultiGPU.git {MULTIGPU_DIR}")
else:
    print("ComfyUI-MultiGPU sudah ada, skip clone.")

run(
    f"python3 -m pip install --no-cache-dir -q "
    f"-r {COMFY_ROOT}/requirements.txt "
    f"-r {MANAGER_DIR}/requirements.txt"
)
shutil.rmtree("/tmp/pip_cache", ignore_errors=True)

# Patch ComfyUI-MultiGPU untuk ComfyUI terbaru + 2x Tesla T4 (P2P=False) + Kaggle 30 GiB RAM (0 Swap)
ds_path = os.path.join(MULTIGPU_DIR, "distorch_2.py")
ds_src = open(ds_path, encoding="utf-8").read()
old_imp = "from comfy.model_management import VRAMState, vram_state, lowvram_available, MIN_WEIGHT_MEMORY_RATIO"
if old_imp in ds_src:
    ds_src = ds_src.replace(
        old_imp,
        "from comfy.model_management import VRAMState, vram_state, MIN_WEIGHT_MEMORY_RATIO\n"
        "            lowvram_available = getattr(mm, 'lowvram_available', True)",
        1,
    )
    open(ds_path, "w", encoding="utf-8").write(ds_src)
    print("Patched distorch_2.py (lowvram_available fallback)")

init_path = os.path.join(MULTIGPU_DIR, "__init__.py")
init_src = open(init_path, encoding="utf-8").read()
old_init = "        exec_device = get_current_device()\n        exec_device = _coerce_torch_device(exec_device)\n"
if old_init in init_src:
    new_init = (
        "        if torch.cuda.is_available() and getattr(tensor_device, 'type', None) == 'cuda':\n"
        "            exec_device = torch.device('cuda', torch.cuda.current_device())\n"
        "        else:\n"
        "            exec_device = _coerce_torch_device(get_current_device())\n"
    )
    open(init_path, "w", encoding="utf-8").write(init_src.replace(old_init, new_init, 1))
    print("Patched __init__.py (DLPack CUDA current_device guard)")

mm_path = os.path.join(MULTIGPU_DIR, "model_management_mgpu.py")
mm_src = open(mm_path, encoding="utf-8").read()
if "CPU_MEMORY_THRESHOLD_PERCENT = 85.0" in mm_src:
    open(mm_path, "w", encoding="utf-8").write(
        mm_src.replace("CPU_MEMORY_THRESHOLD_PERCENT = 85.0", "CPU_MEMORY_THRESHOLD_PERCENT = 99.0", 1)
    )
    print("Patched model_management_mgpu.py (CPU_MEMORY_THRESHOLD_PERCENT=99.0)")

input_dir = os.path.join(COMFY_ROOT, "input")
os.makedirs(input_dir, exist_ok=True)
taxi_path = os.path.join(input_dir, "vintage_yellow_taxi.png")
if not os.path.exists(taxi_path):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Comfy-Org/workflow_templates/refs/heads/main/input/vintage_yellow_taxi.png",
        taxi_path,
    )
    print("Downloaded sample input:", taxi_path)

print("INSTALL & PATCH SELESAI.")

In [ ]:
import base64
import json
import os
import zlib

WF_DIR = "/tmp/ComfyUI/user/default/workflows"
os.makedirs(WF_DIR, exist_ok=True)

WORKFLOWS_B64 = {
    "minimax_h3_image_t2i_dual_gpu.json": "eNrVWdtyo0YQ/RWKhzxhhZsQ8ObsOhtVfNna9eYhLhU1hkFiFwELg2yvy1X5iHxhviQ9w2UAjQRKuWo3fpCZpjk9TJ8+PSM9y1EguzLS7nXfCMyzjUH0M1NV1TObfqj8T5cVOce7qIjSRHZVRY5RQbwkDbBHIezaEEfJF2ZwVPChtwvZvXtmYTRFJk8Zhnifri9uL1MU4PyqjEn07v0nQM9S6mqqiqWuFLmIvmE61FVF06ghjNEaHJ5fFDnN4Uk2iS0EYBdRkpWkChWnPorh6cBL0JZGKxNMqmuY0b6pntObm6tfbmD4EAVrTCDQvi+Epu8nu0kZxy+KINQDjtYb4gUMs402sI4F7LmPxwwgK34nWjsuNogGWoxHrB/px4I1T0tyeFmvbt5eXPKwzbCO1QwpIAXQVhQwy9MM5ySirHiWUfnIuAKpjwH+y+fPP79Jt+HTp+VZhxc7mmvZWKCFEeIAzW3VVM05NkPDCWzfMlVHnQeWjnTNcgINHrgGUkh0VlKY5tLHnz6IGffSLETh7VBcMqLK2yiJtujR2xge/F9jL8sh/4H3YCJ7VqAQE5wUaV5AmACHCLDgyi8D5GryagjIViqgb8o55J4SoscEtxOxTpjbhH6hzKBLqfNkXy7fH64xXZtYZNrEIvPjKBsUWdc0xkDuO074fnFNLKofvJhotnjUetTJZKeS9O9bSQJeCSvp6wNOjF3sGfq916F8sgsz00MPXwdUr12aalKPVRMni3tqmHpJebR+Jam8kozW+Y/zi8OFZNiDQnLEdaRPrKMdwoMy6ljGqNe6/rhEh8XkQasBX+YOyw3F/r4838/6tIZBcxBmmjUg3niPaJPnTgY92AZM3ocB6go9/mYsKdCbNAkiAlu4KFm3LJ5T+TcHNNZtMY+NE/pBvxUIBQ0KQ0hPeOFeBYiIyQkD0xKiVIunciRmKGbcvge6vDp/x2EPlg/QcpsRDtyOa5iPtx+W1+9EVVN7TthLRgHZdDaR9bCZ5/WtcO/IvMbBN2xnwdHb8XH42u20kgeWAed2nYR2LK3IXL9d3i5vrqtVa1TAVOYr4fwvz28v2BSH4xqvHTdIC4Ga+EleqYlPNeTMT3Pcqoc6M+cz9YA2HCkqoUjAsamk80fUS0pDCUkbHMeFD+STyAYRKSqkNU5I/CQVcZoBkPSEiVSQKI6lED5wID1EZAPXOf7nr79zTCWi8OGRCm6Xxj5KIl/CeZmxKFECyFi6R/6XdZ6WSQCPoTgFZAZ0j/IcJ4oUYxTGuCgkkmOoDFnRVN1kH0ekquaw+796sbo2XPaCDZPZqJXNOedjJ6d/4jy9KUlPL81u29dhH2BaQrk0p8plX5jF1sO1AoHGt5j9x8TW4+VovWYRiZa4zYTVTuX3j2ibxQDXLL+mqtCv9M76G7RfWbowAfOJCaBOMV+WZig4zQJnxC3hv8kcTFEIl+A16sN1LMfgLCFcjGDbQKrtBIccWEXiCc1RCFhg0IIWqB4d7x7Mabw3FQRnRQe6Ho5gM69xcD/s1lfYKatfL2/OhdDUa8KsK5oONu8D69iWuuc+Iaa/wUFZFYfANBqt9Z1yXkjSqOgdGBrD2Po1nqftGk7u8LaqvqY8tboj6uia42gL6AP6wrJVCzQmR0mQbqkcKZaiKTKuU1BEFAS66pFuysrC3YME8Sd5GnsoJMAIaKI4h2qFmXVi1bSncklZSr+t6VHI5RNpc+22k2ozA4rWCG9ve/0W+1Q7W+WFw25feXXtYOezJgpvNd9iWDLFQTmCPE89MggOCvYo8ZoTwGAoOB9QAOd1WccXvc2IzQ8VaIeXtVTXGYHD7H4vNHTx2W0xMSXVKWl4apIPnJGcA/mA3R2ml16W4zB65HD7N0YPTcNHTpOSU96n4JivlFKetbEvEIgeeQHsp711Vh77pmC4Gu5BkBf6Hg1V7zRQJhX0CdJfbWhWyp2u6DA2qY0dy8FkKAYzgZjQygGLyRzmzKm75YBbc3aLad7wlsWesCDA3q0Fg7cgUFPdYLRZ3EUnLpQ6m+6CRuaO9LclarKpuUreCt6TngaoGN6tmHiG0bpiP34kOaLrFrBEwuEkppI3003TtOftL1wguWkYFpR3d2e6sZg5tmY6pjE3VNsyFMuY6c4cbPZCVzVHM1dQoTT/9S9iM/PlX4Pp1V8=",
    "minimax_h3_image_edit_dual_gpu.json": "...",
}

for filename, payload in WORKFLOWS_B64.items():
    data = json.loads(zlib.decompress(base64.b64decode(payload)).decode("utf-8"))
    out_path = os.path.join(WF_DIR, filename)
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2, ensure_ascii=False)
    loaders = [
        (n["id"], n["type"], n.get("widgets_values"))
        for n in data["nodes"]
        if "Loader" in n["type"] and n.get("mode", 0) == 0
    ]
    print(f"Installed {out_path} -> active loaders: {loaders}")

In [ ]:
import os
import subprocess

os.chdir("/tmp")
COMFY_ROOT = "/tmp/ComfyUI"

MODELS = [
    (
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/diffusion_models/minimax_h3_image_pruned_w4a8.safetensors?download=true",
        os.path.join(COMFY_ROOT, "models/diffusion_models"),
        "minimax_h3_image_pruned_w4a8.safetensors",
    ),
    (
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors?download=true",
        os.path.join(COMFY_ROOT, "models/text_encoders"),
        "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors",
    ),
    (
        "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/vae/minimax_h3_image_vae_fp16.safetensors?download=true",
        os.path.join(COMFY_ROOT, "models/vae"),
        "minimax_h3_image_vae_fp16.safetensors",
    ),
]

dl_list = "/tmp/models_dl.txt"
to_download = []
for url, folder, fname in MODELS:
    os.makedirs(folder, exist_ok=True)
    p = os.path.join(folder, fname)
    if os.path.exists(p) and not os.path.exists(p + ".aria2") and os.path.getsize(p) > 500 * 1024 * 1024:
        print(f"Sudah ada, skip: {p} ({os.path.getsize(p) / (1024**3):.2f} GiB)")
    else:
        to_download.append((url, folder, fname))

if to_download:
    with open(dl_list, "w", encoding="utf-8") as f:
        for url, folder, fname in to_download:
            f.write(f"{url}\n  dir={folder}\n  out={fname}\n")

    cmd = [
        "aria2c",
        "-i", dl_list,
        "-j", str(len(to_download)),
        "-x", "16",
        "-s", "16",
        "-k", "8M",
        "--file-allocation=none",
        "--allow-overwrite=true",
        "--auto-file-renaming=false",
        "-c",
        "--summary-interval=15",
    ]
    print("Starting parallel multistream aria2c download...", flush=True)
    subprocess.run(cmd, check=True, cwd="/tmp")

print("\n=== Verifikasi File Model ===")
for _, folder, fname in MODELS:
    p = os.path.join(folder, fname)
    size_gb = os.path.getsize(p) / (1024**3)
    assert size_gb > 0.5, f"File tidak valid: {p}"
    print(f"OK {p} -> {size_gb:.2f} GiB")

In [ ]:
import json
import os
import subprocess
import time
import urllib.request

COMFY_ROOT = "/tmp/ComfyUI"
LOG_PATH = "/tmp/comfy_server.log"

subprocess.run(["pkill", "-f", "/tmp/ComfyUI/[m]ain.py"], check=False)
subprocess.run(["pkill", "-f", "python3 [m]ain.py --listen"], check=False)
time.sleep(2)

env = os.environ.copy()
env["PYTHONPATH"] = COMFY_ROOT
env["PYTHONUNBUFFERED"] = "1"
env["TMPDIR"] = "/tmp"
env["XDG_CACHE_HOME"] = "/tmp/.cache"
env["HF_HOME"] = "/tmp/.cache/huggingface"
env["TORCH_HOME"] = "/tmp/.cache/torch"
env["PATH"] = f"/opt/bin:/usr/local/nvidia/bin:/usr/local/cuda/bin:{env.get('PATH', '')}"
env["LD_LIBRARY_PATH"] = f"/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/nvidia/lib:{env.get('LD_LIBRARY_PATH', '')}"

log_f = open(LOG_PATH, "w", encoding="utf-8")
proc = subprocess.Popen(
    [
        "python3",
        "main.py",
        "--listen", "0.0.0.0",
        "--port", "8188",
        "--cuda-device", "all",
        "--preview-method", "auto",
        "--enable-cors-header", "*",
    ],
    cwd=COMFY_ROOT,
    env=env,
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"ComfyUI starting PID={proc.pid}, log={LOG_PATH}", flush=True)

for i in range(90):
    if proc.poll() is not None:
        log_f.flush()
        print("ComfyUI EXITED prematurely! Last 60 lines:")
        print("".join(open(LOG_PATH, encoding="utf-8").readlines()[-60:]))
        raise RuntimeError(f"ComfyUI exited with code {proc.returncode}")
    try:
        stats = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2).read())
        wfs = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/api/userdata?dir=workflows", timeout=2).read())
        print(f"ComfyUI ONLINE in {i * 2}s!")
        print("ComfyUI version:", stats["system"]["comfyui_version"], "| PyTorch:", stats["system"]["pytorch_version"])
        for dev in stats["devices"]:
            print("Device:", dev["name"], f"({dev['vram_total'] / (1024**3):.2f} GiB total)")
        print("Workflows terdeteksi:", wfs)
        break
    except Exception:
        time.sleep(2)
else:
    print("Timeout menunggu ComfyUI! Tail log:")
    print("".join(open(LOG_PATH, encoding="utf-8").readlines()[-60:]))
    raise TimeoutError("ComfyUI gagal startup dalam 180 detik")

In [ ]:
import os
import re
import shutil
import stat
import subprocess
import time
import urllib.request
try:
    from IPython.display import HTML, display
except ImportError:
    HTML = display = None

CF_BIN = shutil.which("cloudflared") or "/tmp/cloudflared"
if not os.path.exists(CF_BIN):
    print("Mengunduh cloudflared ke /tmp/cloudflared ...", flush=True)
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        CF_BIN,
    )
    os.chmod(CF_BIN, os.stat(CF_BIN).st_mode | stat.S_IXUSR | stat.S_IXGRP | stat.S_IXOTH)

subprocess.run(["pkill", "-f", "[c]loudflared tunnel.*127.0.0.1:8188"], check=False)
time.sleep(1)

cf_log = "/tmp/cf_tunnel.log"
log_f = open(cf_log, "w", encoding="utf-8")
cf_proc = subprocess.Popen(
    [CF_BIN, "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8188"],
    cwd="/tmp",
    stdout=log_f,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)
print(f"Cloudflared PID={cf_proc.pid} started...", flush=True)

public_url = None
for _ in range(30):
    time.sleep(1)
    txt = open(cf_log, encoding="utf-8", errors="ignore").read()
    m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", txt)
    if m:
        public_url = m.group(0)
        break

if not public_url:
    print("Gagal menangkap URL trycloudflare! Log:")
    print(open(cf_log, encoding="utf-8", errors="ignore").read()[-1500:])
    raise RuntimeError("Cloudflare quick tunnel gagal dibuat")

# Tunggu DNS propagasi sebentar (jangan query terlalu cepat agar DNS container tidak negatif-cache NXDOMAIN)
time.sleep(6)
ready_http = False
for _ in range(10):
    try:
        with urllib.request.urlopen(f"{public_url}/system_stats", timeout=5) as r:
            if r.status == 200:
                ready_http = True
                break
    except Exception:
        time.sleep(2)

print(f"\nComfyUI Public URL: {public_url} (probe_ok={ready_http})")
if display and HTML:
    display(HTML(f'<h3><a href="{public_url}" target="_blank" rel="noopener noreferrer">Buka ComfyUI: {public_url}</a></h3>'))

In [ ]:
# Opsional: jalankan 1x smoke test Text-to-Image (1024x1024, 6 step) via API & cek VRAM/RAM
import json
import os
import shutil
import subprocess
import threading
import time
import urllib.request
try:
    from IPython.display import Image as IPyImage, display
except ImportError:
    IPyImage = display = None

NVIDIA_SMI = shutil.which("nvidia-smi") or "/opt/bin/nvidia-smi"
peaks = {"gpu0": 0, "gpu1": 0, "ram_gib": 0.0}
stop_ev = threading.Event()


def monitor():
    while not stop_ev.is_set():
        try:
            out = subprocess.check_output(
                [NVIDIA_SMI, "--query-gpu=index,memory.used", "--format=csv,noheader,nounits"],
                text=True,
                timeout=3,
            ).strip().splitlines()
            for line in out:
                idx, mem = [int(x.strip()) for x in line.split(",")]
                if idx == 0:
                    peaks["gpu0"] = max(peaks["gpu0"], mem)
                elif idx == 1:
                    peaks["gpu1"] = max(peaks["gpu1"], mem)
            for cg in ("/sys/fs/cgroup/memory.current", "/sys/fs/cgroup/memory/memory.usage_in_bytes"):
                if os.path.exists(cg):
                    peaks["ram_gib"] = max(peaks["ram_gib"], int(open(cg).read().strip()) / (1024**3))
                    break
        except Exception:
            pass
        stop_ev.wait(1.0)


prompt_graph = {
    "1": {"class_type": "UNETLoaderMultiGPU", "inputs": {"unet_name": "minimax_h3_image_pruned_w4a8.safetensors", "weight_dtype": "default", "device": "cuda:1"}},
    "2": {"class_type": "CLIPLoaderMultiGPU", "inputs": {"clip_name": "qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "type": "minimax", "device": "cuda:0"}},
    "3": {"class_type": "VAELoaderMultiGPU", "inputs": {"vae_name": "minimax_h3_image_vae_fp16.safetensors", "device": "cuda:1"}},
    "4": {"class_type": "MiniMaxH3ImageConditioning", "inputs": {"clip": ["2", 0], "vae": ["3", 0], "prompt": "visualization of a hellscape that is gently sloping yet still filled with fire—reminiscent of a volcanic eruption in the background—along with barren, leafless trees.", "width": 1024, "height": 1024}},
    "5": {"class_type": "ConditioningZeroOut", "inputs": {"conditioning": ["4", 0]}},
    "6": {"class_type": "KSampler", "inputs": {"model": ["1", 0], "positive": ["4", 0], "negative": ["5", 0], "latent_image": ["4", 1], "seed": 199175202768065, "steps": 6, "cfg": 1.0, "sampler_name": "euler", "scheduler": "simple", "denoise": 1.0}},
    "7": {"class_type": "VAEDecode", "inputs": {"samples": ["6", 0], "vae": ["3", 0]}},
    "8": {"class_type": "SaveImage", "inputs": {"images": ["7", 0], "filename_prefix": "smoke_test_t2i"}},
}

t = threading.Thread(target=monitor, daemon=True)
t.start()
t0 = time.time()
req = urllib.request.Request(
    "http://127.0.0.1:8188/prompt",
    data=json.dumps({"prompt": prompt_graph}).encode("utf-8"),
    headers={"Content-Type": "application/json"},
)
pid = json.loads(urllib.request.urlopen(req, timeout=30).read())["prompt_id"]
print(f"Queued smoke test prompt_id={pid}", flush=True)

img_path = None
while True:
    time.sleep(2)
    hist = json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}", timeout=10).read())
    if pid in hist:
        status = hist[pid]["status"]
        dt = time.time() - t0
        print(f"Selesai dalam {dt:.2f}s | status={status.get('status_str')}")
        if status.get("status_str") != "success":
            raise RuntimeError(json.dumps(status.get("messages", []), indent=2))
        fname = hist[pid]["outputs"]["8"]["images"][0]["filename"]
        img_path = os.path.join("/tmp/ComfyUI/output", fname)
        break

stop_ev.set()
t.join(timeout=2)
print(f"Peak VRAM: GPU0={peaks['gpu0']} MiB, GPU1={peaks['gpu1']} MiB | Peak cgroup RAM={peaks['ram_gib']:.2f} GiB")
if img_path and os.path.exists(img_path):
    print("Output image:", img_path)
    if display and IPyImage:
        display(IPyImage(filename=img_path, width=512))